# Sizing the bias_trim Output

Adapted from `macros/ota_fcasc_fd/scripts/sizing/lookup_commands.ipynb`, which is the overview of the pygmid lookup commands this notebook uses.

**Copyright 2026 Michael Koefinger**

SPDX-License-Identifier: Apache-2.0 WITH SHL-2.1

bias_trim was sized by hand equations and simulation sweeps, so this notebook rebuilds the derivation after the fact. It starts from the spec (200 nA at code 4, output = 8.33 nA x (16 + 2 x code), the OTA bias pin at 0.35 to 0.8 V, supply 1.45 to 1.55 V), derives the unit, the groups and the switch with gm/ID lookups, then reads the drawn sizes from the schematic and compares. Nothing here changes the schematic. Where the derivation disagrees with the drawing, the cell says so and leaves the drawing alone.

Every number below is a gm/ID model estimate from the tt tables at 300 K, with no mismatch, no process corner and no layout wiring. The simulated values are in `macros/bias_trim/README.md` and `macros/bias_gm/README.md`.

The cell is a PMOS mirror output on the gate net `vz` of the `bias_gm` core. Each group is a bank of unit PMOS devices with the sources on `vdd`, in series with a switch of the same device on the drain side. The trim bits go straight to the switch gates.

In [1]:
# Packages
from pygmid import Lookup as lk
import numpy as np
import re
from pathlib import Path

In [2]:
# Spec and the choices the README gives
VDD_MIN, VDD, VDD_MAX = 1.45, 1.5, 1.55
I_OUT   = 200e-9      # A, output at code 4
CODE    = 4
N_FIX   = 16          # fixed units
BITS    = (16, 8, 4, 2)   # units behind bit 3 down to bit 0
N_CORE  = 24          # PMOS units per core leg in bias_gm
V_PIN   = (0.35, 0.8) # V, OTA bias pin range
V_PIN_T = 0.6         # V, typical level
UNIT_P  = (0.5, 5.0)  # um, W and L of the core PMOS unit
DROP_README = 15e-3   # V, drain rise of the 16-unit groups in the README
CELL    = 'bias_trim'
DATA    = Path('data')
SCH     = Path('../schematic') / (CELL + '.sch')
SCH_GM  = Path('../schematic/bias_gm.sch')

### Read the sizes out of the schematic

The schematic is the source of truth, so nothing here restates a drawn size. The checks are the invariants the topology rests on: the unit device equals the core PMOS unit of `bias_gm`, every switch element is that same device, and the group sizes are the binary weights.

In [3]:
def num(s):
    return float(s[:-1]) if s.endswith('u') else float(s) * 1e6

def read_sizes(path):
    txt = path.read_text()
    dev = {}
    for m in re.finditer(r'^C \{([^}]+)\.sym\} \S+ \S+ \S+ \S+ \{name=(\S+)\n(.*?)\}', txt, re.S | re.M):
        a = dict(re.findall(r'^(\w+)=(\S+)$', m.group(3), re.M))
        if 'w' in a and 'l' in a:
            dev[m.group(2)] = dict(w=num(a['w']), l=num(a['l']), m=int(a.get('m', 1)), sym=m.group(1))
    return dev

D = read_sizes(SCH)
G = read_sizes(SCH_GM)
for n, d in D.items():
    print(f"{n:4s} {d['sym']:14s} W {d['w']:5.2f} um  L {d['l']:5.2f} um  m {d['m']:3d}")

UNITS = ('MPF', 'MP3', 'MP2', 'MP1', 'MP0')
SWS   = ('MSF', 'MS3', 'MS2', 'MS1', 'MS0')
assert all((D[n]['w'], D[n]['l']) == UNIT_P for n in UNITS + SWS), 'unit or switch is not the core unit'
assert (G['M4']['w'], G['M4']['l']) == UNIT_P and G['M4']['m'] == N_CORE
assert [D[n]['m'] for n in UNITS] == [N_FIX, *BITS]
assert all(D[n]['m'] == 2 for n in SWS)

MPF  sg13_lv_pmos   W  0.50 um  L  5.00 um  m  16
MSF  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MP3  sg13_lv_pmos   W  0.50 um  L  5.00 um  m  16
MS3  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MP2  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   8
MS2  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MP1  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   4
MS1  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MP0  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MS0  sg13_lv_pmos   W  0.50 um  L  5.00 um  m   2
MPD  sg13_lv_pmos   W  0.50 um  L  5.00 um  m  40


In [4]:
tab = lk(str(DATA / 'sg13cmos5l_lv_pmos.mat'))
UT = 1.380649e-23 * 300 / 1.602177e-19

def per_um(f, L, vgs, vds, vsb=0.0):
    return float(tab.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb)) / float(tab['W'])

def ratio(f, L, vgs, vds, vsb=0.0):
    return float(tab.lookup(f, L=L, VGS=vgs, VDS=vds, VSB=vsb))

def vsg_for(idw, L, vsd):
    return float(tab.lookupVGS(ID_W=idw, L=L, VDS=vsd, VSB=0))

### The unit

The output is a plain mirror of the core, so the unit is the core PMOS unit and the current per unit follows from the spec: 200 nA at code 4 is 16 + 2 x 4 = 24 units, 8.33 nA each, which is also the 24 legs of the core mirror in `bias_gm`.

In [5]:
N_OUT = N_FIX + 2 * CODE
I_U = I_OUT / N_OUT
IDW = I_U / UNIT_P[0]
print(f'{N_OUT} units at code {CODE}, {I_U*1e9:.2f} nA per unit, {IDW*1e9:.1f} nA/um, core mirror has {G["M4"]["m"]} legs')
vsg = vsg_for(IDW, UNIT_P[1], VDD - V_PIN_T)
print(f'unit at the typical pin: VSG {vsg*1e3:.1f} mV, gm/ID {ratio("GM_ID", UNIT_P[1], vsg, VDD - V_PIN_T):.1f} 1/V, VDSS {ratio("VDSS", UNIT_P[1], vsg, VDD - V_PIN_T)*1e3:.0f} mV')

24 units at code 4, 8.33 nA per unit, 16.7 nA/um, core mirror has 24 legs
unit at the typical pin: VSG 340.3 mV, gm/ID 24.6 1/V, VDSS 111 mV


### Trim range and step

The range has to cover what the core does over process. From the `bias_gm` README the core current spreads from 0.78 (rhigh wcs) to 1.31 (rhigh bcs), 0.79 to 1.22 for the LV/HV skew, and 0.678 to 1.367 for global plus local Monte Carlo (all multiples of 200 nA at tt, code 4 not applied). The units needed to bring each case back to 200 nA are 24 over the multiple, and the code is that count minus 16, over 2.

In [6]:
cases = {'rhigh wcs': 0.78, 'rhigh bcs': 1.31, 'LV/HV skew low': 0.78, 'LV/HV skew high': 1.21,
         'Monte Carlo low': 0.678, 'Monte Carlo high': 1.367}
print('case                 core/200nA  units needed  code needed')
for k, mult in cases.items():
    n = N_OUT / mult
    print(f'{k:20s} {mult:9.3f} {n:12.1f} {(n - N_FIX)/2:12.1f}')

n_bits = 4
lo, hi = N_FIX, N_FIX + 2 * (2 ** n_bits - 1)
print(f'4 bits, step 2 units: {lo} to {hi} units, core multiples {N_OUT/hi:.2f} to {N_OUT/lo:.2f}')
hi3 = N_FIX + 2 * (2 ** 3 - 1)
print(f'3 bits would stop at {hi3} units, core multiple {N_OUT/hi3:.2f}, short of rhigh wcs (needs {N_OUT/0.78:.1f}) and Monte Carlo (needs {N_OUT/0.678:.1f})')

case                 core/200nA  units needed  code needed
rhigh wcs                0.780         30.8          7.4
rhigh bcs                1.310         18.3          1.2
LV/HV skew low           0.780         30.8          7.4
LV/HV skew high          1.210         19.8          1.9
Monte Carlo low          0.678         35.4          9.7
Monte Carlo high         1.367         17.6          0.8
4 bits, step 2 units: 16 to 46 units, core multiples 0.52 to 1.50
3 bits would stop at 30 units, core multiple 0.80, short of rhigh wcs (needs 30.8) and Monte Carlo (needs 35.4)


Four bits are needed because the upper cases ask for 31 to 35 units and three bits stop at 30. The fixed 16 units set the low end: 16 units cover cores up to 1.5 times nominal, above the 1.37 of the Monte Carlo.

The step is two units, so the step in percent depends on the code, and the residual after the best code is uniform within half a step.

In [7]:
print('code  units  step/%   (ideal, 2 units over the code)')
for c in (0, 4, 15):
    n = N_FIX + 2 * c
    print(f'{c:4d} {n:6d} {200/n:7.2f}')
step4 = 2 / N_OUT
print(f'residual at code 4: rms {100*step4/np.sqrt(12):.2f} %, max {100*step4/2:.2f} %   (README Monte Carlo: rms 2.4 %, max 4.7 %)')

code  units  step/%   (ideal, 2 units over the code)
   0     16   12.50
   4     24    8.33
  15     46    4.35
residual at code 4: rms 2.41 %, max 4.17 %   (README Monte Carlo: rms 2.4 %, max 4.7 %)


The residual matches the README Monte Carlo (2.4 % rms). The step from code 0 to 1 is two units over 16, 12.5 %, and the step into code 15 two units over 44, 4.5 %. The simulated steps (4.46 to 12.50 %) agree. The README row quoted 6.25 % and 2.1 % as ideal and was corrected on 2026-11-06.

### Output resistance of the unit bank

Output resistance of the bank at the worst pin voltage, from the unit GDS at VDD 1.45 V and a pin at 0.8 V (the switch drop is added below). The span is the current change over the 0.35 to 0.8 V pin range.

In [8]:
L = UNIT_P[1]
vsd_w = VDD_MIN - V_PIN[1]
vsg_w = vsg_for(IDW, L, vsd_w)
gds_u = per_um('GDS', L, vsg_w, vsd_w) * UNIT_P[0]
print('code  units   rout/GOhm   span 0.45 V/%   README sim worst rout GOhm / span %')
sim = {0: '0.38 / 0.844', 4: '0.25 / 0.879', 15: '0.12 / 0.930'}
for c in (0, 4, 15):
    n = N_FIX + 2 * c
    r = 1 / (n * gds_u)
    i = n * I_U
    print(f'{c:4d} {n:6d} {r/1e9:11.3f} {100*(V_PIN[1]-V_PIN[0])/r/i:14.3f}   {sim[c]}')

code  units   rout/GOhm   span 0.45 V/%   README sim worst rout GOhm / span %
   0     16       0.393          0.859   0.38 / 0.844
   4     24       0.262          0.859   0.25 / 0.879
  15     46       0.137          0.859   0.12 / 0.930


The model gives 0.26 GOhm at code 4 against the simulated 0.25 GOhm (ff, 125 C, smallest average r_out). The simulated values at code 0 and 15 are worse because the hot leakage of the open groups and the switch drops add to the table value. The span in percent is nearly code-independent in the model (current and conductance both scale with the unit count).

### Switch on the drain side

The switch sits between the unit drain and `iout`. Its gate is at vss for a closed group, so its source (the unit drain) carries V_SG equal to the drain voltage, which is the pin voltage plus its own drop. That is 0.35 V at the bottom of the pin range, close to the LV PMOS threshold, so the switch is not in strong inversion everywhere. The drop is solved from the large-signal table: find V_SD such that the current of the m = 2 element equals the group current, with V_SG = pin + V_SD and V_SB = VDD - pin - V_SD.

In [9]:
def sw_drop(n_units, pin, m=2, vdd=VDD):
    # drop over a closed switch element of m devices carrying n_units * I_U
    i = n_units * I_U
    vsd = np.arange(0.0, 0.6, 0.001)
    idv = np.array([ratio('ID', L, pin + v, v, vdd - pin - v) for v in vsd]) / float(tab['W']) * UNIT_P[0] * m
    return float(np.interp(i, idv, vsd)) if i <= idv[-1] else np.nan

print('group units   ' + '   '.join(f'pin {p:.2f} V' for p in (0.35, 0.5, 0.6, 0.8)))
for n in (16, 8, 4, 2):
    print(f'{n:11d}   ' + '   '.join(f'{1e3*sw_drop(n, p):8.1f} mV' for p in (0.35, 0.5, 0.6, 0.8)))

group units   pin 0.35 V   pin 0.50 V   pin 0.60 V   pin 0.80 V


         16      245.5 mV      105.7 mV       53.5 mV       21.8 mV


          8      202.4 mV       68.3 mV       29.5 mV       11.3 mV


          4      167.4 mV       42.2 mV       16.0 mV        5.8 mV


          2      138.4 mV       24.3 mV        8.5 mV        2.9 mV


In [10]:
# Ron as drop over current, the slope of the switch at the group current
print('group units   ' + '   '.join(f'pin {p:.2f} V' for p in (0.35, 0.5, 0.6, 0.8)))
for n in (16, 8, 4, 2):
    print(f'{n:11d}   ' + '   '.join(f'{sw_drop(n, p)/(n*I_U)/1e3:8.0f} kOhm' for p in (0.35, 0.5, 0.6, 0.8)))

group units   pin 0.35 V   pin 0.50 V   pin 0.60 V   pin 0.80 V


         16       1841 kOhm        793 kOhm        401 kOhm        163 kOhm


          8       3035 kOhm       1024 kOhm        442 kOhm        169 kOhm


          4       5021 kOhm       1266 kOhm        479 kOhm        173 kOhm


          2       8306 kOhm       1458 kOhm        511 kOhm        174 kOhm


At the top of the pin range (0.8 V) the 16-unit groups drop 22 mV and the 8, 4 and 2 unit groups 11, 5.8 and 2.9 mV in the model. The README rows read 17 mV, 8.6, 4.4 and 2.2 mV at ss, -40 C, pin 0.8 V, and quote 15 mV, so the model is about 30 % high against those rows. The drop is linear in the group current as in the README rows, Ron of the element is 163 to 174 kOhm at 0.8 V (README rows: about 130 kOhm).

The drop grows steeply towards the low end of the pin range. At 0.6 V the 16-unit groups drop 54 mV, at 0.35 V they drop 246 mV (Ron 0.4 MOhm at 0.6 V, 1.8 to 8.3 MOhm at 0.35 V) because the switch is near threshold there, with V_SB of 1.1 V. The README quotes the drop only at 0.8 V. The unit stays saturated (V_SD above 0.85 V against a V_DSS of 111 mV), and the drop reaches the output only through the unit conductance, shown next.

In [11]:
print('effect of the drop on the unit current, ddrop x gds_unit / I_unit')
for pin in (0.35, 0.6, 0.8):
    vsd = VDD_MIN - pin
    vs = vsg_for(IDW, L, vsd)
    gds = per_um('GDS', L, vs, vsd) * UNIT_P[0]
    for n in (16, 2):
        d = sw_drop(n, pin)
        print(f'pin {pin:.2f} V, {n:2d}-unit group: drop {d*1e3:6.1f} mV -> {100*gds*d/I_U:5.2f} % of the unit current')
print('README: 15 mV on the 16-unit groups moves the unit current by about 0.2 to 0.4 % (sim, worst corner)')

effect of the drop on the unit current, ddrop x gds_unit / I_unit
pin 0.35 V, 16-unit group: drop  245.5 mV ->  0.27 % of the unit current


pin 0.35 V,  2-unit group: drop  138.4 mV ->  0.15 % of the unit current
pin 0.60 V, 16-unit group: drop   53.5 mV ->  0.08 % of the unit current


pin 0.60 V,  2-unit group: drop    8.5 mV ->  0.01 % of the unit current


pin 0.80 V, 16-unit group: drop   21.8 mV ->  0.04 % of the unit current
pin 0.80 V,  2-unit group: drop    2.9 mV ->  0.01 % of the unit current
README: 15 mV on the 16-unit groups moves the unit current by about 0.2 to 0.4 % (sim, worst corner)


The table gives 0.04 % for the 22 mV at 0.8 V and 0.27 % for the 246 mV at 0.35 V. At 0.8 V this is 5 to 10 times below the README's 0.2 to 0.4 % for 15 mV, and the model does not explain the larger figure. The README number is a sim value at a worst corner and stands, the model is a lower bound. Adding the 0.27 % to the 0.86 % table span of the bank gives about 1.1 % over the pin range, against the simulated 0.88 %.

Why one switch element of 2 devices per group and not one switch per unit. Per unit device at 8.33 nA the drop is small, but 46 switches against 10 cost 90 um2 of gate area. The drop of the m = 2 element carrying the 16-unit group, against 16 single-unit switches (each carrying one unit current at its own V_SD):

In [12]:
print('pin/V   m=2 element, 16 units   single unit switch (m=1, 1 unit)   gate area 10 vs 46 switches')
for p in (0.35, 0.6, 0.8):
    print(f'{p:5.2f} {1e3*sw_drop(16, p):17.1f} mV {1e3*sw_drop(1, p, m=1):27.1f} mV')
a_dev = UNIT_P[0] * UNIT_P[1]
print(f'gate area: 10 switch devices {10*a_dev:.0f} um2, 46 {46*a_dev:.0f} um2')

pin/V   m=2 element, 16 units   single unit switch (m=1, 1 unit)   gate area 10 vs 46 switches


 0.35             245.5 mV                       138.4 mV


 0.60              53.5 mV                         8.5 mV


 0.80              21.8 mV                         2.9 mV
gate area: 10 switch devices 25 um2, 46 115 um2


### Leakage of an open switch

The table covers V_GS from 0 to 1.5 V, so only the off state with V_SG = 0 is available. That is the state of an open group, since its unit drain floats up to `vdd` (README: within 0.5 mV) and the switch gate is at `vdd`. The leakage is the table current at V_GS = 0 and V_SD = VDD - pin. Negative V_GS, as with a pin above the gate drive, is not in the table.

In [13]:
print('pin/V   leakage per m=1 switch device / pA   code 0: 8 open devices / pA')
for p in (0.35, 0.6, 0.8):
    i0 = per_um('ID', L, 0.0, VDD - p, 0.0) * UNIT_P[0]
    print(f'{p:5.2f} {i0*1e12:30.3f} {8*i0*1e12:30.3f}')
print('README ff/125 C, di_ena high, code 0, 30 units behind open switches: 0.86 nA')

pin/V   leakage per m=1 switch device / pA   code 0: 8 open devices / pA
 0.35                          0.221                          1.764
 0.60                          0.220                          1.763
 0.80                          0.220                          1.762
README ff/125 C, di_ena high, code 0, 30 units behind open switches: 0.86 nA


The table value is hundreds of times below the README's 0.86 nA. The table is tt at 300 K and covers the channel only. The README leakage is at ff, 125 C and is dominated by the junction leakage of the floating drains, which the tables do not hold. The model cannot predict it and the sim value is the one that counts for the 6.7 nA at code 4 in the `di_ena` low state.

### Mismatch

The tables carry no Pelgrom coefficient (their `INFO` field only names the PDK and the PSP model), so no mismatch estimate is made here. The matching of the 24 core legs against the 24 output units comes from the Monte Carlo of the README: 1.39 % sigma of the test output over `ibias_200n` at code 4, 1.60 % for the whole output and 1.76 % with only `bias_gm` and `bias_trim` mismatched (200 runs, tt, sample noise about 8 %).

### Comparison with the drawing

In [14]:
print(f"{'group':14s} {'derived':>8s} {'drawn':>6s}   basis")
exp = {'MPF': N_FIX, 'MP3': 16, 'MP2': 8, 'MP1': 4, 'MP0': 2}
for n in UNITS:
    print(f"{n + ' units':14s} {exp[n]:8d} {D[n]['m']:6d}   " + ('fixed units, set by the low end of the range' if n == 'MPF' else f'binary weight 2 x {exp[n]//2}'))
print(f"{'total units':14s} {N_FIX + sum(BITS):8d} {sum(D[n]['m'] for n in UNITS):6d}   16 + 2 x 15")
print(f"{'output @ code 4':14s} {N_FIX + 2*CODE:8d} {N_FIX + 2*CODE:6d}   {I_U*(N_FIX + 2*CODE)*1e9:.0f} nA")
for n in SWS:
    print(f"{n + ' devices':14s} {2:8d} {D[n]['m']:6d}   one element of 2 units per group (house rule m >= 2)")
print(f"{'unit W/L':14s} {'0.5/5':>8s} {str(D['MP1']['w']) + '/' + str(D['MP1']['l']):>6s}   core PMOS unit of bias_gm")
a = lambda names: sum(D[n]['m'] for n in names) * UNIT_P[0] * UNIT_P[1]
print(f"gate area: units {a(UNITS):.0f} um2, switches {a(SWS):.0f} um2, MPD dummy m={D['MPD']['m']} {a(('MPD',)):.0f} um2")

group           derived  drawn   basis
MPF units            16     16   fixed units, set by the low end of the range
MP3 units            16     16   binary weight 2 x 8
MP2 units             8      8   binary weight 2 x 4
MP1 units             4      4   binary weight 2 x 2
MP0 units             2      2   binary weight 2 x 1
total units          46     46   16 + 2 x 15
output @ code 4       24     24   200 nA
MSF devices           2      2   one element of 2 units per group (house rule m >= 2)
MS3 devices           2      2   one element of 2 units per group (house rule m >= 2)
MS2 devices           2      2   one element of 2 units per group (house rule m >= 2)
MS1 devices           2      2   one element of 2 units per group (house rule m >= 2)
MS0 devices           2      2   one element of 2 units per group (house rule m >= 2)
unit W/L          0.5/5 0.5/5.0   core PMOS unit of bias_gm
gate area: units 115 um2, switches 25 um2, MPD dummy m=40 100 um2


All unit counts, the switch element and the unit device follow from the spec.

The README ideal steps (6.25 % and 2.1 %, now 12.5 % and 4.5 %) and the cell area (145 um2 with a 5 um2 dummy, now 240 um2 with `MPD` m=40 at 100 um2) were corrected on 2026-11-06 after this notebook. Still open, the design is unchanged. The model switch drop at 0.8 V is 22 mV against the README's 15 mV, and it grows to 246 mV at a pin of 0.35 V, which the README does not list (unit current effect 0.27 %, within the simulated 0.88 % span only if the two do not add). The README's 0.2 to 0.4 % current shift from a 15 mV drop is 5 to 10 times the table estimate.